# 01 V2 — Common Crawl Query (English-Speaking Web)

Collects candidate URLs from Common Crawl across the full English-speaking web.
Uses URL-path heuristics to surface pages likely to represent specific schema.org types.

## Scope
- TLDs: `.com`, `.co.uk`, `.org`, `.net`, `.ie`, `.ca`, `.com.au`, `.co.nz`
- Language: English only (`languages = 'eng'`)
- Page types: business homepages + typed sub-pages (`/menu`, `/events`, `/recipe`, etc.)

## Options
- **Option A** (recommended): DuckDB over CC S3 parquet — free, no AWS needed, runs anywhere
- **Option B**: AWS Athena — faster for very large queries, ~$1.50/query

## Output
`data/raw/cc_candidate_urls.jsonl` — `{url, probable_type, tld, crawl}`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

DATA_DIR = Path('../data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

CRAWL = 'CC-MAIN-2026-08'   # update to latest: https://commoncrawl.org/connect/blog
OUT_PATH = DATA_DIR / 'cc_candidate_urls.jsonl'

print(f'Crawl: {CRAWL}')
print(f'Output: {OUT_PATH}')

## Option A: DuckDB over CC S3 Parquet (Recommended)

Queries the CC columnar index directly from S3 — no download required, no AWS account needed.
Fastest from EC2 `us-east-1` but works anywhere. Typical runtime: 2-10 min per query.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("""
    SET s3_region = 'us-east-1';
    SET s3_access_key_id = '';
    SET s3_secret_access_key = '';
    SET s3_url_style = 'path';
""")

PARQUET = f"s3://commoncrawl/cc-index/table/cc-main/warc/crawl={CRAWL}/subset=warc/*.parquet"

# Verify connectivity with a tiny query
test = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{PARQUET}', hive_partitioning=1)
    WHERE url_host_tld = 'ie' AND fetch_status = 200
    LIMIT 1
""").fetchone()
print('Connection OK, test count:', test)

In [ ]:
# ---------------------------------------------------------------------------
# URL-path patterns → probable @type
# ---------------------------------------------------------------------------
# Each entry: (url_pattern, probable_type, max_rows)
# Patterns target sub-pages that strongly signal a schema type.
# Adjust max_rows to control dataset size per type.
# ---------------------------------------------------------------------------

TYPE_QUERIES = [
    # --- Typed sub-pages (high precision) ---
    ("url LIKE '%/menu%' OR url LIKE '%/our-menu%'",         'Restaurant',    5_000),
    ("url LIKE '%/recipe%' OR url LIKE '%/recipes/%'",       'Recipe',        5_000),
    ("url LIKE '%/events/%' OR url LIKE '%/whats-on%'",      'Event',         5_000),
    ("url LIKE '%/jobs/%' OR url LIKE '%/careers/%'",        'JobPosting',    5_000),
    ("url LIKE '%/faq%' OR url LIKE '%/faqs%'",              'FAQPage',       5_000),
    ("url LIKE '%/accommodation%' OR url LIKE '%/rooms/%'",  'Hotel',         3_000),
    ("url LIKE '%/course%' OR url LIKE '%/courses/%'",       'Course',        3_000),
    ("url LIKE '%/product/%' OR url LIKE '%/products/%'",    'Product',      10_000),
    ("url LIKE '%/blog/%' OR url LIKE '%/article/%'",        'Article',       8_000),
    # --- Business homepages (broad) ---
    ("url_path = '/' OR url LIKE '%/about%' OR url LIKE '%/services%'",
                                                             'LocalBusiness', 20_000),
]

# TLDs we care about
ENGLISH_TLDS = "('com', 'org', 'net', 'ie', 'ca', 'uk', 'au', 'nz', 'us', 'biz', 'info')"

print('Query plan:')
for pattern, ptype, limit in TYPE_QUERIES:
    print(f'  {ptype:15s} — limit {limit:,}')

In [ ]:
from tqdm.notebook import tqdm
import time

all_results = []

for url_pattern, probable_type, max_rows in tqdm(TYPE_QUERIES, desc='Querying CC'):
    sql = f"""
        SELECT
            url,
            url_host_registered_domain AS domain,
            url_host_tld               AS tld,
            languages
        FROM read_parquet('{PARQUET}', hive_partitioning=1)
        WHERE url_host_tld IN {ENGLISH_TLDS}
          AND fetch_status  = 200
          AND (languages = 'eng' OR languages IS NULL)
          AND ({url_pattern})
        USING SAMPLE {max_rows} ROWS
    """
    try:
        rows = con.execute(sql).fetchall()
        for url, domain, tld, lang in rows:
            all_results.append({
                'url': url,
                'domain': domain,
                'tld': tld,
                'probable_type': probable_type,
                'source': 'cc',
                'crawl': CRAWL,
            })
        print(f'  {probable_type}: {len(rows):,} rows')
    except Exception as e:
        print(f'  {probable_type}: FAILED — {e}')
    time.sleep(1)  # be polite

print(f'\nTotal CC candidates: {len(all_results):,}')

In [ ]:
from collections import Counter

# Deduplicate by URL
seen = set()
deduped = []
for r in all_results:
    if r['url'] not in seen:
        seen.add(r['url'])
        deduped.append(r)

print(f'After dedup: {len(deduped):,} unique URLs')
print('\nType breakdown:')
for t, n in Counter(r['probable_type'] for r in deduped).most_common():
    print(f'  {t:20s} {n:6,}')
print('\nTLD breakdown:')
for t, n in Counter(r['tld'] for r in deduped).most_common(10):
    print(f'  .{t:10s} {n:6,}')

In [ ]:
# Save
with open(OUT_PATH, 'w') as f:
    for r in deduped:
        f.write(json.dumps(r) + '\n')

size_mb = OUT_PATH.stat().st_size / 1e6
print(f'Saved {len(deduped):,} records → {OUT_PATH} ({size_mb:.1f} MB)')
print('\nNext: 02_v2_wdc_download.ipynb')

## Option B: AWS Athena

Use for larger queries or when you need faster execution. ~$1.50/query.
Requires `boto3` and a configured AWS account with Athena + the CC Glue catalog set up.

In [ ]:
# Equivalent Athena query — uncomment and run if you have AWS credentials

ATHENA_QUERY = f"""
SELECT
    url,
    url_host_registered_domain AS domain,
    url_host_tld               AS tld,
    CASE
        WHEN url LIKE '%/menu%' OR url LIKE '%/our-menu%'           THEN 'Restaurant'
        WHEN url LIKE '%/recipe%' OR url LIKE '%/recipes/%'         THEN 'Recipe'
        WHEN url LIKE '%/events/%' OR url LIKE '%/whats-on%'        THEN 'Event'
        WHEN url LIKE '%/jobs/%' OR url LIKE '%/careers/%'          THEN 'JobPosting'
        WHEN url LIKE '%/faq%' OR url LIKE '%/faqs%'                THEN 'FAQPage'
        WHEN url LIKE '%/accommodation%' OR url LIKE '%/rooms/%'    THEN 'Hotel'
        WHEN url LIKE '%/course%' OR url LIKE '%/courses/%'         THEN 'Course'
        WHEN url LIKE '%/product/%' OR url LIKE '%/products/%'      THEN 'Product'
        WHEN url LIKE '%/blog/%' OR url LIKE '%/article/%'          THEN 'Article'
        ELSE 'LocalBusiness'
    END AS probable_type
FROM ccindex.ccindex
WHERE crawl       = '{CRAWL}'
  AND subset      = 'warc'
  AND url_host_tld IN ('com','org','net','ie','ca','uk','au','nz')
  AND fetch_status = 200
  AND languages   = 'eng'
  AND (
        url LIKE '%/menu%'          OR url LIKE '%/recipe%'
     OR url LIKE '%/events/%'       OR url LIKE '%/jobs/%'
     OR url LIKE '%/faq%'           OR url LIKE '%/accommodation%'
     OR url LIKE '%/course%'        OR url LIKE '%/product/%'
     OR url LIKE '%/blog/%'         OR url LIKE '%/about%'
     OR url LIKE '%/services%'
  )
TABLESAMPLE BERNOULLI(5)   -- 5% random sample, tune as needed
"""

print('Athena query ready.')
print('To run: boto3 Athena client, s3://your-bucket/results/ as output location.')
# result = run_athena_query(ATHENA_QUERY, 's3://your-bucket/athena-results/')